# CFNA autopilot — one model, trains itself, saves itself

Single-model training that **survives you walking away**: it checkpoints to
Google Drive every 15 minutes on a background thread, so when Colab reclaims
the runtime (idle timeout, 12h cap, disconnect) you lose at most 15 minutes,
with **zero manual steps**. Reopen later, run cells 1-4 again, it resumes
exactly where it stopped.

**Pick your target in cell 3.** Honest guidance for a 100 GPU-hour budget:

| Preset | Params | 100h verdict | Needs |
|---|---|---|---|
| `large_337m` | ~337M | **RECOMMENDED** — fully trainable to a strong model | A100 or L4 |
| `xl_1b` | ~1.2B | reachable but **under-trained** at 100h (~12B/20B tokens); a well-trained 337M usually beats it | **A100-40GB only** |
| `base_90m` | ~92M | very safe, fast, fits anything incl. T4 | any GPU |

Chinchilla, in one line: *a fully-trained small model beats a half-trained
big one at equal compute.* 337M is the compute-optimal pick here. `xl_1b` is
wired and ready if you want the parameter count anyway.

In [ ]:
# 1) Setup
import os
REPO = "LMMinier/nueronce"; BRANCH = "claude/cfna-neural-core-verify-ldvgn3"
if not os.path.exists("nueronce"):
    !git clone --branch $BRANCH https://github.com/$REPO nueronce
%cd nueronce
!git pull
%pip install -q numpy 'datasets>=2.19,<4' pytest cryptography cffi
import torch
gb = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
print(torch.__version__, torch.cuda.get_device_name(0) if gb else 'NO GPU', f'{gb:.0f}GB')

In [ ]:
# 2) Mount Drive + restore any prior progress (safe to run every session)
from google.colab import drive
from pathlib import Path
import shutil
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/CFNA_autopilot"); DRIVE.mkdir(parents=True, exist_ok=True)
Path("checkpoints").mkdir(exist_ok=True)
for f in DRIVE.glob("*"):
    if f.is_file(): shutil.copy2(f, Path("checkpoints")/f.name); print("restored:", f.name)

In [ ]:
# 3) CHOOSE TARGET + budget. Auto-picks batch to the GPU; edit PRESET to taste.
PRESET = "large_337m"      # or "xl_1b" (A100-40GB only) or "base_90m"
CKPT   = f"checkpoints/cfna_{PRESET}.pt"
LR     = "3e-4"            # ladder: drop to 1e-4 then 3e-5 as it plateaus
SESSION_MIN = 690          # ~11.5h; safely under Colab's 12h cap

import torch
gb = torch.cuda.get_device_properties(0).total_memory/1e9
if PRESET == "xl_1b":
    assert gb >= 38, f"xl_1b needs A100-40GB; this GPU has {gb:.0f}GB. Use large_337m."
    BATCH, ACCUM = 4, 4   # tight on A100-40GB; raise ACCUM if OOM
elif PRESET == "large_337m":
    BATCH, ACCUM = (16, 2) if gb >= 38 else (6, 4)
else:
    BATCH, ACCUM = (24, 1) if gb >= 22 else (12, 2)
print(f"target {PRESET} | GPU {gb:.0f}GB -> batch {BATCH} x{ACCUM}accum | lr {LR}")

In [ ]:
# 4) Corpus (rebuild each fresh runtime; HF-cached so reruns are fast).
#    Bigger corpus = lower floor. 337M wants multi-GB; 1B wants more.
TARGET_BYTES = "4000000000" if PRESET in ("large_337m", "xl_1b") else "1500000000"
!python scripts/dump_corpus_stack.py --out corpus_train --phase 2 \
    --target-bytes $TARGET_BYTES --val-every 20
!du -sh corpus_train

In [ ]:
# 5) LAUNCH AUTOPILOT: background training + Drive auto-save every 15 min.
#    Re-running this cell resumes (never restarts). Safe to close the tab on
#    Colab Pro+; on regular Pro keep it open. Either way Drive stays current.
import os, subprocess, threading, time, shutil
from pathlib import Path
cmd = ["python", "-u", "scripts/train_checkpoint.py", "--preset", PRESET,
       "--corpus", "corpus_train", "--minutes", str(SESSION_MIN), "--seq", "192",
       "--batch", str(BATCH), "--grad-accum", str(ACCUM), "--lr", LR, "--amp",
       "--save-every-min", "5", "--resume", "--out", CKPT]
Path("logs").mkdir(exist_ok=True); LOG = Path("logs/autopilot.log")
PROC = subprocess.Popen(cmd, stdout=LOG.open("w"), stderr=subprocess.STDOUT,
                        text=True, start_new_session=True)
def _autosave():
    while PROC.poll() is None:
        time.sleep(900)  # 15 min
        for p in list(Path('checkpoints').glob(f'cfna_{PRESET}.pt*')):
            try: shutil.copy2(p, DRIVE/p.name)
            except Exception as e: print('save retry:', e)
        print('drive-saved', time.strftime('%H:%M'), flush=True)
threading.Thread(target=_autosave, daemon=True).start()
print(f"AUTOPILOT running (PID {PROC.pid}). Drive auto-save every 15 min -> {DRIVE}")
print("You can close this tab (Pro+) or leave it open (Pro). It resumes on rerun.")

In [ ]:
# 5a) Monitor — steps, epochs, shards, tokens, bpb, ETA (rerun anytime)
import re, json
lines = LOG.read_text(errors='replace').splitlines()
print('alive:', PROC.poll() is None); print('\n'.join(lines[-6:]))
b = [float(m.group(1)) for l in lines if (m := re.search(r'held-out bpb ([0-9.]+)', l))]
st = [int(m.group(1)) for l in lines if (m := re.search(r'step\s+(\d+)', l))]
if b:
    seen = (st[-1]*BATCH*ACCUM*191)/1e9 if st else 0
    print(f"\nbpb {b[0]:.3f} -> {b[-1]:.3f} (best {min(b):.3f}) | ~{seen:.2f}B tokens seen")
    print('plateau? drop LR 3x in cell 3 and rerun cell 5 for the next ladder rung.')

In [ ]:
# 5b) MAKE IT AN ASSISTANT — conversational + code SFT (run once base bpb is
#     low, ~<=1.6). Builds ONE mixed dataset: dialogue/facts/reasoning
#     (conversational, the majority) PLUS instruction->code pairs mined from
#     permissive source, capped at <=25% per register so neither dominates.
#     The cloned repo is Apache-2.0 = safe code; add more permissive .py trees
#     to --code-dirs for scale.
!git clone --depth 1 https://github.com/psf/requests _perm_requests 2>/dev/null || true  # Apache-2.0
CODE_DIRS = "nueronce,_perm_requests"
!python scripts/build_conversation_sft.py --out-dir data/assistant_sft --code-dirs $CODE_DIRS
!python scripts/train_conversation.py --data data/assistant_sft \
    --preset $PRESET --init-from $CKPT --loss response \
    --out-dir checkpoints/assistant --minutes 120 --batch 16 --lr 1e-4 --amp
import shutil; from pathlib import Path
for p in Path("checkpoints/assistant").glob("*.pt"): shutil.copy2(p, DRIVE/f"assistant_{p.name}")
print("assistant SFT done + backed up to Drive")

In [ ]:
from pathlib import Path
# 6) Chat with the current best (run anytime; loads from local checkpoint)
from cfna.chat import Conversation, load_checkpoint
m, ck = load_checkpoint("checkpoints/assistant/best.pt" if Path("checkpoints/assistant/best.pt").exists() else CKPT)
convo = Conversation(m, system="You are CFNA.", temperature=0.3)
for q in ["Hello! Who are you?", "Write a Python function that reverses a string.",
          "What is the capital of France?"]:
    print(f"User: {q}\nCFNA: {convo.say(q)}\n")

## The budget, by epochs / shards / tokens (why this is safe)

The trainer is **step-based, not epoch-based** — it samples random `seq x
batch` windows from the sharded corpus, so 'epoch' = corpus_bytes / (steps x
tokens/step). What matters is **tokens seen**, shown live in cell 5a.

- tokens/step = batch x grad_accum x seq (e.g. 16 x 2 x 192 = 6,144)
- 100 A100-hours at ~1 step/s (337M) x 6,144 = **~2.2B tokens** — enough to
  train 337M well (Chinchilla-optimal ~7B, so a few 100h blocks converge it).
- the corpus is auto-sharded by `dump_corpus_stack`; the trainer streams all
  shards via `ByteCorpus`, so you never manage shards by hand.

**Speed levers already wired:** AMP (fp16) ~2x; `--grad-accum` for a bigger
effective batch without more memory; `--compile` (add to cell 5's `cmd` for
torch.compile) ~1.2-1.8x after warmup; batch auto-sized to the GPU. Biggest
single win is simply a **faster GPU** (A100 ~ 5x a T4) — pick it in Runtime.

**Reaching 1B:** set `PRESET='xl_1b'` in cell 3 on an A100-40GB. It will
train and save exactly the same way; just expect it under-trained at 100h and
re-run across several sessions (autopilot makes that painless).